# 02 — Preprocessing

**Purpose:** Load and filter EEG, extract presentation segments, and generate all regressors.

Output of this notebook:
- `presentations` — list of 5 EEG arrays (one per video repeat), shape `(n_samples, n_channels)`
- `X` — regressor matrix, shape `(n_samples, n_features)`
- `feat_names` — names of each feature column

These are passed directly into the experiment notebook.

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np
import matplotlib.pyplot as plt
print('Ready.')

## Step 1 — Load EEG with bandpass filter

The EEG is filtered to 0.1–30 Hz before resampling:
- **0.1 Hz highpass** removes slow DC drift that would corrupt the TRF
- **30 Hz lowpass** removes muscle noise above the range we care about
- Filter is applied first, then resampled from 512 Hz → 64 Hz

This takes ~25 seconds the first time.

In [ ]:
from sign_language_trf.config import load_session_config
from sign_language_trf.preprocessing.eeg_loader import load_eeg
from sign_language_trf.preprocessing.synchronization import (
    extract_presentation_segments, extract_segment_eeg
)

cfg = load_session_config('config/session.yaml')

print('Loading EEG (applying 0.1–30 Hz bandpass filter)...')
eeg = load_eeg(cfg.eeg_path, cfg, bandpass=True, l_freq=0.1, h_freq=30.0, verbose=False)
print(f'Done. {len(eeg.ch_names)} channels @ {eeg.sfreq} Hz, {eeg.duration_s:.0f}s')

## Step 2 — Extract presentation segments

The EEG recording contains 5 repetitions of the same ASL video.
Each repetition is bracketed by trigger codes 255 (onset) and 254 (end).
We extract each bracket as a separate array — these become our 5 CV folds.

In [ ]:
segs = extract_presentation_segments(eeg, onset_code=255, offset_code=254)
presentations = [extract_segment_eeg(eeg, seg) for seg in segs]

print(f'Extracted {len(presentations)} presentation segments:')
for i, P in enumerate(presentations):
    print(f'  Pres {i+1}: {P.shape[0]} samples × {P.shape[1]} channels  ({P.shape[0]/eeg.sfreq:.1f}s)')

## Step 3 — Load supporting data for regressors

Beyond the basic motion CSV, we also load:
- **ELAN annotations** — sign onset/offset times for linguistically-grounded regressors
- **Raw Rokoko data** — 3D joint positions for acceleration and spatial regressors

In [ ]:
from sign_language_trf.preprocessing.motion_loader import load_motion
from sign_language_trf.preprocessing.markers_loader import load_markers
from sign_language_trf.preprocessing.elan_loader import load_elan
from sign_language_trf.preprocessing.raw_rokoko_loader import load_raw_rokoko
import glob

motion  = load_motion(cfg.motion_path)
markers = load_markers(cfg.markers_path)

elan_file = glob.glob('sign_language_trf/data/elan/*.eaf')[0]
elan = load_elan(elan_file)
print(f'ELAN: {elan.n_annotations} signs loaded')

raw_rokoko_files = glob.glob('sign_language_trf/data/rokoko/*raw*.csv')
if raw_rokoko_files:
    raw_rokoko = load_raw_rokoko(raw_rokoko_files[0])
    print(f'Raw Rokoko: {raw_rokoko.participant}, {raw_rokoko.n_frames} frames')
else:
    raw_rokoko = None
    print('Raw Rokoko: not found (acceleration regressors unavailable)')

## Step 4 — Build regressor matrix

The `RegressorManager` resamples all signals to 64 Hz (the common analysis rate)
and applies kind-aware normalization:
- **Continuous regressors** (velocity, motion energy, stasis) → z-scored to zero mean, unit variance
- **Event regressors** (onset/offset impulses) → kept as raw 0/1

This normalization means TRF weights are in comparable units across features.

In [ ]:
from sign_language_trf.regressors.manager import (
    RegressorManager, register_elan_regressors, register_raw_regressors
)

mgr = RegressorManager(motion, markers, target_fs=float(cfg.common_rate))

# Register ELAN-based linguistic regressors
register_elan_regressors(elan)

# Register raw Rokoko regressors (acceleration, position) if available
if raw_rokoko is not None:
    register_raw_regressors(raw_rokoko)

print(f'\nAll available regressors ({len(mgr.available)} total):')
for name in mgr.available:
    print(f'  {name}')

## Step 5 — Inspect key regressors

This plot shows the main kinematic signals over the first 60 seconds.
- **motion_energy**: overall movement magnitude (how much Tristan is moving)
- **velocity_stasis**: hold/pause state (high = hands are still, low = hands moving)
- **sign_active**: are we inside an ELAN-annotated sign? (1 = yes, 0 = inter-sign gap)

In [ ]:
fs   = float(cfg.common_rate)
t_end = 60  # seconds to show
n    = int(t_end * fs)
t    = np.arange(n) / fs

me     = mgr.get('motion_energy')[:n]
stasis = mgr.get('velocity_stasis')[:n]

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(t, me, color='#2c7bb6', lw=0.8)
axes[0].set_ylabel('Motion Energy (z-scored)', fontsize=10)
axes[0].set_title('Kinematic regressors — first 60 seconds', fontweight='bold')
axes[0].axhline(0, color='gray', lw=0.5, ls='--')

axes[1].plot(t, stasis, color='#d7191c', lw=0.8)
axes[1].set_ylabel('Velocity Stasis (z-scored)', fontsize=10)
axes[1].set_xlabel('Time (s)', fontsize=10)

# Overlay sign boundaries from ELAN
for ann in elan.annotations:
    if ann.onset_s <= t_end:
        for ax in axes:
            ax.axvline(ann.onset_s, color='green', lw=0.4, alpha=0.4)

from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color='green', alpha=0.4, label='ELAN sign onset')],
               fontsize=8, loc='upper right')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
print('Green lines = ELAN sign onsets. Notice signs cluster at motion peaks.')

## Step 6 — Build X matrix for selected experiment

The experiment config specifies which regressors to use.
The cell below builds the final X matrix — rows = time samples, columns = features.
This X is trimmed to match the shortest presentation length.

In [ ]:
from sign_language_trf.config import load_experiment_config

# Change this to switch experiments (must match a file in config/experiments/)
EXPERIMENT = 'motion_state_v3'

exp_cfg = load_experiment_config(f'config/experiments/{EXPERIMENT}.yaml')
X, feat_names = mgr.get_matrix(exp_cfg.regressors)

# Trim X to match EEG presentation length
n_trim = min(X.shape[0], min(P.shape[0] for P in presentations))
X = X[:n_trim]
presentations_trim = [P[:n_trim] for P in presentations]

print(f'Experiment    : {EXPERIMENT}')
print(f'Regressors    : {feat_names}')
print(f'X shape       : {X.shape}  (samples × features)')
print(f'EEG shape     : {presentations_trim[0].shape}  per presentation')
print(f'\nAll shapes match — ready for modeling.')